# ==============================================================================
# 🧱 MATERIALS SCIENCE: CONCRETE COMPRESSIVE STRENGTH VIA SUPPORT VECTOR REGRESSOR
# ==============================================================================
### Core Principles:
Predicting compressive strength (MPa) of high-performance concrete from constituent mixture proportions (Cement, Slag, Fly Ash, Water, Superplasticizer, Aggregates, Age).

Hydration chemical kinetics generate highly non-linear, smooth response surfaces where RBF SVR excels over piecewise tree step-functions.


In [1]:
# STEP 1: ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Materials Science Environment ready.")


✅ STEP 1: Materials Science Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET CLEANUP
Loading Kaggle Concrete Compressive Strength dataset (1,030 mix formulations).


In [2]:
# STEP 2 & 3: LOAD & CLEANUP
data_path = 'data/concrete/Concrete Compressive Strength.csv'
df = pd.read_csv(data_path)

df.columns = [c.strip().lower().replace(' ', '_').replace('(', '').replace(')', '') for c in df.columns]
print(f"📊 Dataset Shape: {df.shape[0]} mixtures, {df.shape[1]} physical parameters")

target_col = [c for c in df.columns if 'strength' in c][0]
print(f"Target Attribute: '{target_col}'")
print(f"Strength Summary (MPa):\n{df[target_col].describe().round(2)}")
df.head(3)


📊 Dataset Shape: 1030 mixtures, 9 physical parameters
Target Attribute: 'concrete_compressive_strength'
Strength Summary (MPa):
count    1030.00
mean       35.82
std        16.71
min         2.33
25%        23.71
50%        34.44
75%        46.14
max        82.60
Name: concrete_compressive_strength, dtype: float64


,cement,blast_furnace_slag,fly_ash,water,superplasticizer,coarse_aggregate,fine_aggregate,age_day,concrete_compressive_strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.986111
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.887366
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.269535


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features and target compressive strength.


In [3]:
# STEP 4 & 5: SEGREGATION
X = df.drop(columns=[target_col])
y = df[target_col]

print(f"Mix Components: {X.shape[1]}")
print(f"Missing values: {X.isnull().sum().sum()}")


Mix Components: 8
Missing values: 0


## 🔒 STEP 6: TRAIN-TEST SPLIT
80/20 train/test split.


In [4]:
# STEP 6: SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Train Formulations: {X_train.shape[0]}")
print(f"Test Formulations : {X_test.shape[0]}")


Train Formulations: 824
Test Formulations : 206


## ⚔️ STEP 7 & 8: ALGORITHM DUEL (Decision Tree vs RBF SVR)
Contrasting discontinuous axis-aligned tree splits against continuous RBF Gaussian manifolds.


In [5]:
# STEP 7 & 8: ALGORITHM DUEL
pipe_tree = DecisionTreeRegressor(max_depth=6, random_state=42)
pipe_tree.fit(X_train, y_train)

pipe_svr = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(kernel='rbf', C=50, epsilon=1.0))
])
pipe_svr.fit(X_train, y_train)

r2_tree = r2_score(y_test, pipe_tree.predict(X_test))
r2_svr = r2_score(y_test, pipe_svr.predict(X_test))

print("="*65)
print(f"Decision Tree Regressor R² : {r2_tree*100:.2f}% | MAE: {mean_absolute_error(y_test, pipe_tree.predict(X_test)):.2f} MPa")
print(f"RBF SVR Pipeline R²        : {r2_svr*100:.2f}% | MAE: {mean_absolute_error(y_test, pipe_svr.predict(X_test)):.2f} MPa")
print("="*65)


Decision Tree Regressor R² : 73.75% | MAE: 6.29 MPa


RBF SVR Pipeline R²        : 86.35% | MAE: 4.26 MPa


## 🎯 STEP 9: TUNING HYPERPARAMETERS VIA 5-FOLD CV
Optimizing $C$, $\varepsilon$, and $\gamma$.


In [6]:
# STEP 9: GRID SEARCH
param_grid = {
    'svr__C': [10, 50, 100, 200],
    'svr__epsilon': [0.5, 1.0, 2.0],
    'svr__gamma': ['scale', 0.05, 0.1, 0.2]
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
pipe_tune = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(kernel='rbf'))
])

grid = GridSearchCV(pipe_tune, param_grid=param_grid, cv=cv, scoring='r2', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Concrete Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV R²: {grid.best_score_*100:.2f}%")


🏆 Best Concrete Hyperparameters: {'svr__C': 200, 'svr__epsilon': 2.0, 'svr__gamma': 0.1}
🎯 Best 5-Fold CV R²: 86.87%


## 📊 STEP 10 & 11: MATERIALS DIAGNOSTIC REPORT & RESIDUALS
Evaluating compressive prediction accuracy and safety tolerance.


In [7]:
# STEP 10 & 11: EVALUATION REPORT
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
sv_count = len(best_model.named_steps['svr'].support_)

print("📋 CIVIL ENGINEERING SPECIFICATION METRICS:")
print(f"   Mean Absolute Error (MAE) : {mae:.2f} MPa")
print(f"   Root Mean Squared Error   : {rmse:.2f} MPa")
print(f"   R² Coefficient            : {r2 * 100:.2f}%")
print(f"   Active Support Vectors    : {sv_count}/{len(X_train)} ({(sv_count/len(X_train))*100:.1f}%)")


📋 CIVIL ENGINEERING SPECIFICATION METRICS:
   Mean Absolute Error (MAE) : 4.15 MPa
   Root Mean Squared Error   : 5.82 MPa
   R² Coefficient            : 86.87%
   Active Support Vectors    : 520/824 (63.1%)


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & BATCH PLANT SMOKE TEST
Deploying the pipeline and testing batch plant automated curing quality check.


In [8]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/concrete_strength_svr_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_mix = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_strength = loaded_pipe.predict(sample_mix)[0]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🏗️ LIVE BATCHING PLANT QUALITY AUDIT:")
print(f"   Forecast Compressive Strength: 🧱 {pred_strength:.2f} MPa")
print(f"   Actual Lab Test Strength     : {y_test.iloc[0]:.2f} MPa")
print(f"   Mix Audit Latency            : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/concrete_strength_svr_pipeline.joblib (17,154 bytes)

🏗️ LIVE BATCHING PLANT QUALITY AUDIT:
   Forecast Compressive Strength: 🧱 54.01 MPa
   Actual Lab Test Strength     : 52.91 MPa
   Mix Audit Latency            : 2.973 ms (SLA < 2.0ms: CHECK)
